# Evaluate WavLM LA anti-spoofing on ASVspoof 2019 LA

Scores the checkpoint from `01_train_wavlm_la.ipynb` on official **LA dev** (or **eval**).

Paper reference: **0.45% EER** on ASVspoof 2019 LA  
https://doi.org/10.1145/3708597.3708621

## How to read the numbers

- **Oracle EER** — best threshold on *this* split (usual reporting).
- **Train-val threshold** — threshold from speaker-held-out train val (stricter transfer).
- Compare only to other **LA** runs (LFCC-LA, zero-shot replay-on-LA). Not to PA / ASVspoof 2017 replay EER.

We are unlikely to hit 0.45% with a frozen encoder + 4s crops + a small head. Treat the paper figure as an upper-bound reference, not a pass/fail gate.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "eval_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\wavlm_la2019")
sys.path.insert(0, str(ROOT))

import json
import torch
from experiment_lib import DEFAULT_CKPT, PAPER_LA_EER_PERCENT, RUNS_DIR
from eval_lib import eval_wavlm_la

print("cuda:", torch.cuda.is_available())
print("checkpoint:", DEFAULT_CKPT, DEFAULT_CKPT.exists())
print("paper LA EER %:", PAPER_LA_EER_PERCENT)

cuda: True
checkpoint: D:\speaker-verification-system\replay-cnn-baseline\experiments\wavlm_la2019\runs\wavlm_la\best_wavlm_la2019.pt True
paper LA EER %: 0.45


## Eval knobs

- `SPLIT = "dev"` for the usual development protocol.
- `SPLIT = "eval"` for official eval (larger, slower).
- `MAX_UTTS = 500` for a quick check after smoke training.

In [2]:
SPLIT = "dev"
MAX_UTTS = 0
BATCH_SIZE = 2
FORCE_CPU = False

## Score LA split

Writes ROC, confusion matrix, metrics JSON, and per-utt CSV under `runs/eval_wavlm_la_<split>/`.

In [3]:
results = eval_wavlm_la(
    checkpoint=DEFAULT_CKPT,
    output_dir=RUNS_DIR / f"eval_wavlm_la_{SPLIT}",
    split=SPLIT,
    batch_size=BATCH_SIZE,
    max_utts=MAX_UTTS,
    force_cpu=FORCE_CPU,
)
{
    "split": results["split"],
    "n": results["num_scored_files"],
    "oracle_eer_percent": results["metrics_at_oracle_eer_threshold"]["eer_percent"],
    "ckpt_thr_eer_percent": results["metrics_at_train_val_threshold"]["eer_percent"],
    "paper_la_eer_percent": results["paper_la_eer_percent"],
}

Loaded D:\speaker-verification-system\replay-cnn-baseline\experiments\wavlm_la2019\runs\wavlm_la\best_wavlm_la2019.pt
model=microsoft/wavlm-base; device=cuda; thr=0.5532; epoch=6


LA dev readability:   0%|          | 0/24844 [00:00<?, ?it/s]

LA2019 dev: {'total': 24844, 'bonafide': 2548, 'spoof': 22296, 'speakers': 20} (skipped=0)


Scoring:   0%|          | 0/12422 [00:00<?, ?it/s]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


KeyboardInterrupt: 

## Side-by-side with LFCC-LA (if trained)

Both models are LA-trained. Lower oracle EER is better.

In [ ]:
lfcc_metrics = (
    ROOT.parent / "lfcc_la2019" / "runs" / f"eval_lfcc_la_{SPLIT}" / f"la2019_{SPLIT}_metrics.json"
)
wavlm_metrics = RUNS_DIR / f"eval_wavlm_la_{SPLIT}" / f"la2019_{SPLIT}_metrics.json"

if lfcc_metrics.exists():
    lfcc = json.loads(lfcc_metrics.read_text(encoding="utf-8"))
    print("LFCC-LA oracle EER %:", lfcc.get("oracle_eer_percent"))
else:
    print("No LFCC-LA eval at", lfcc_metrics)

wavlm = json.loads(wavlm_metrics.read_text(encoding="utf-8"))
print("WavLM-LA oracle EER %:", wavlm["metrics_at_oracle_eer_threshold"]["eer_percent"])
print("Paper LA EER %:", PAPER_LA_EER_PERCENT)